In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import re

In [ ]:
# Find the two most recent runs for each algorithm
build_dir = Path('../build')

def find_recent_files(pattern):
    files = sorted(build_dir.glob(pattern), key=lambda x: x.stat().st_mtime, reverse=True)
    return files[:2] if len(files) >= 2 else files

dijkstra_files = find_recent_files('*dijkstra*.csv')
deltastep_files = find_recent_files('*deltastep*.csv')
bellmanford_files = find_recent_files('*bellmanford*.csv')

print(f"Dijkstra files: {[f.name for f in dijkstra_files]}")
print(f"Deltastep files: {[f.name for f in deltastep_files]}")
print(f"BellmanFord files: {[f.name for f in bellmanford_files]}")

In [ ]:
# Extract query hash from filename
def get_query_hash(filename):
    match = re.search(r'_(\w{8})_', filename)
    return match.group(1) if match else None

# Find matching files by query hash
dijkstra_file = None
deltastep_file = None
bellmanford_file = None

for dijk_f in dijkstra_files:
    dijk_hash = get_query_hash(dijk_f.name)
    for delta_f in deltastep_files:
        delta_hash = get_query_hash(delta_f.name)
        if dijk_hash and delta_hash and dijk_hash == delta_hash:
            # Check if we also have a matching bellmanford file
            for bellman_f in bellmanford_files:
                bellman_hash = get_query_hash(bellman_f.name)
                if bellman_hash and bellman_hash == dijk_hash:
                    dijkstra_file = dijk_f
                    deltastep_file = delta_f
                    bellmanford_file = bellman_f
                    break
            if dijkstra_file:
                break
    if dijkstra_file:
        break

# Fallback: use most recent files if no hash match
if not dijkstra_file or not deltastep_file or not bellmanford_file:
    raise ValueError("No match found")

print(f"Using: {dijkstra_file.name}")
print(f"Using: {deltastep_file.name}")
print(f"Using: {bellmanford_file.name}")

In [ ]:
# Load data
df_dijkstra = pd.read_csv(dijkstra_file)
df_deltastep = pd.read_csv(deltastep_file)
df_bellmanford = pd.read_csv(bellmanford_file)

print(f"Dijkstra: {len(df_dijkstra)} queries")
print(f"Deltastep: {len(df_deltastep)} queries")
print(f"BellmanFord: {len(df_bellmanford)} queries")

In [ ]:
# Check for distance errors
df_merged = pd.merge(
    df_dijkstra,
    df_deltastep,
    on=['from_node_id', 'to_node_id', 'rank'],
    suffixes=('_dijkstra', '_deltastep')
)

df_merged = pd.merge(
    df_merged,
    df_bellmanford,
    on=['from_node_id', 'to_node_id', 'rank']
)
df_merged.rename(columns={'distance': 'distance_bellmanford', 'time': 'time_bellmanford'}, inplace=True)

errors_dijkstra_deltastep = df_merged[df_merged['distance_dijkstra'] != df_merged['distance_deltastep']]
errors_dijkstra_bellmanford = df_merged[df_merged['distance_dijkstra'] != df_merged['distance_bellmanford']]

if len(errors_dijkstra_deltastep) > 0:
    print(f"⚠️  Found {len(errors_dijkstra_deltastep)} distance mismatches between Dijkstra and Delta-stepping!")
    print(errors_dijkstra_deltastep[['from_node_id', 'to_node_id', 'distance_dijkstra', 'distance_deltastep']].head(10))
else:
    print("✓ All distances match between Dijkstra and Delta-stepping")

if len(errors_dijkstra_bellmanford) > 0:
    print(f"⚠️  Found {len(errors_dijkstra_bellmanford)} distance mismatches between Dijkstra and Bellman-Ford!")
    print(errors_dijkstra_bellmanford[['from_node_id', 'to_node_id', 'distance_dijkstra', 'distance_bellmanford']].head(10))
else:
    print("✓ All distances match between Dijkstra and Bellman-Ford")

In [ ]:
# Histogram comparing execution times
fig, ax = plt.subplots(figsize=(10, 6))

bin_width = np.max([df_dijkstra['time'].max(), df_deltastep['time'].max(), df_bellmanford['time'].max()])/20

sns.histplot(df_dijkstra['time'], label='Dijkstra', alpha=0.5, ax=ax, binwidth=bin_width)
sns.histplot(df_deltastep['time'], label='Delta-stepping', alpha=0.5, ax=ax, binwidth=bin_width)
sns.histplot(df_bellmanford['time'], label='Bellman-Ford', alpha=0.5, ax=ax, binwidth=bin_width)

ax.set_xlabel('Time (μs)')
ax.set_ylabel('Count')
ax.set_title('Execution Time Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Box plot by query rank
plot_data = []
for _, row in df_merged.iterrows():
    plot_data.append({'rank': row['rank'], 'time': row['time_dijkstra']/10**6, 'algorithm': 'Dijkstra'})
    plot_data.append({'rank': row['rank'], 'time': row['time_deltastep']/10**6, 'algorithm': 'Delta-stepping'})
    plot_data.append({'rank': row['rank'], 'time': row['time_bellmanford']/10**6, 'algorithm': 'Bellman-Ford'})

df_plot = pd.DataFrame(plot_data)

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_plot, x='rank', y='time', hue='algorithm', ax=ax)
ax.set_xlabel('Query Rank Bin')
ax.set_ylabel('Time (s)')
plt.yscale('log')
ax.set_title('Execution Time by Query Rank')
plt.tight_layout()
plt.show()